# TP 1: LDA/QDA y optimización matemática de modelos

Se puede consultar la introducción teórica en el mono-notebook, se prefiere mantener este lo más chico posible.

In [1]:
# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA, FasterQDA, EfficientQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.bench import Benchmark
from utils.datasets import (get_iris_dataset, get_letters_dataset, 
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)


### Respuesta a consigna 5

Sean $A = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$, $B = \begin{bmatrix} x & y \\ z & w \end{bmatrix}$ y por lo tanto $B^T = \begin{bmatrix} x & z \\ y & w \end{bmatrix}$.

$$A \times B^T = \begin{bmatrix} (ax + by) & (az + bw) \\ (cx + dy) & (cz + dw) \end{bmatrix}$$

$$diag(A \times B^T) = \begin{bmatrix} ax + by \\ cz + dw \end{bmatrix}$$

Por otro lado, el producto punto entre A y B es:

$$A \cdot B = \begin{bmatrix} ax & by \\ cz & dw \end{bmatrix}$$

y sumando sobre las columnas:

$$\sum_{cols} (A \cdot B) = \begin{bmatrix} ax + by \\ cz + dw \end{bmatrix}$$

Ambos resultados coinciden, por lo tanto $diag(A \times B^T) = \sum_{cols} (A \cdot B)$.

## Comparación benchmarks

In [2]:
X_full, y_full = get_penguins_dataset()

X_full.shape, y_full.shape

((342, 4), (342, 1))

In [3]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['Adelie'],
        ['Adelie'],
        ['Adelie'],
        ['Adelie'],
        ['Adelie']], dtype=object),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [4]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 50,
    warmup = 10,
    mem_runs = 5,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 65
Warmup runs: 10
Peak Memory usage runs: 5
Running time runs: 50
Train size rows (approx): 240
Test size rows (approx): 102
Test size fraction: 0.3


In [5]:
# bencheamos un par
to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/5 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/5 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/5 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/5 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/50 [00:00<?, ?it/s]

In [6]:
# hacemos un summary
summ = b.summary()
summ[['train_median_ms', 'test_median_ms', 'train_mem_median_mb', 'test_mem_median_mb']]

,train_median_ms,test_median_ms,train_mem_median_mb,test_mem_median_mb
model,,,,
QDA,0.208688,2.806417,0.011147,0.004740
TensorizedQDA,0.214729,1.071372,0.011421,0.004669
FasterQDA,0.199433,0.051893,0.010925,0.264641
EfficientQDA,0.200796,0.039461,0.011238,0.041153


## Conclusiones (consigna #7)
- `train_median_ms`: es prácticamente el mismo para todos los modelos, lo cual está bien porque el código es el mismo.
- `test_median_ms`: esta sección es una de las más importantes, acá sí podemos apreciar cómo varía el tiempo de ejecución. Más específicamente, se puede apreciar cómo fuimos reduciendo considerablemente y, por ende, mejorando en cada modelo.
- `test_mem_median_mb`: esta sección también es de las más importantes, acá podemos apreciar cómo cambiar el consumo de memoria para cada modelo. A resaltar, FasterQDA consume una mucho mayor que sus pares: 56 veces más que TensorizedQDA. Esto es muy importante y a considerar para dataset de gran tamaño. En cuanto a EfficientQDA, podemos ver que consume más memoria que TensorizedQDA (8x), pero al mismo tiempo es considerablemente más rápido (24x)

En líneas generales, podemos observar cómo EfficientQDA ofrece una muy buena relación en cuanto a tiempo de procesamiento y memoria utilizada. Para el modelo de FasterQDA, que al eliminar el loop en la predicción y utilizar todo el dataset para el cálculo, logramos mejorar considerablemente el tiempo de ejecución, pero a costa de un consumo de memoria muy elevado.

Las observaciones condicen con lo esperado y sorprende el hecho de las grandes mejoras que se logra aplicando soluciones con un trasfondo matemático.